https://www.earthdata.nasa.gov/data/catalog/ornl-cloud-main-melt-onset-dates-1841-1.0#documents-and-resources

In [ ]:
import xarray as xr
import rioxarray as rxr
from global_snowmelt_runoff_onset.config import Config, Tile
import easysnowdata
import glob
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import contextily as ctx
from matplotlib_scalebar.scalebar import ScaleBar

import matplotlib.gridspec as gridspec
import matplotlib.patheffects as path_effects
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import ConnectionPatch
from matplotlib_scalebar.scalebar import ScaleBar
import shapely

In [2]:
from pathlib import Path

config = Config('config/global_config_v9.txt')

# Figures are scoped by dataset version so a v10 run cannot overwrite the v9 figures.
# Switching versions = editing the config path above.
VERSION = config.version
FIGURE_DIR = Path('figures') / VERSION / 'passive_comparison'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SAS token is valid until 2026-07-03 00:51 UTC (221.6 hours)
----------------------------------------
Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
season

In [ ]:
passive_files = glob.glob(f"data/Main_Melt_Onset_Dates_1841/data/*1.tif")
years = [int(f.split('.')[2][:4]) for f in passive_files]
ds_list = []
da_template = rxr.open_rasterio(passive_files[0], masked=True).squeeze()
for f, year in zip(passive_files, years):
    da = rxr.open_rasterio(f, masked=True).squeeze()
    da = da.rename('melt_onset_dayofyear')
    da = da.rio.reproject_match(da_template)
    da = da.assign_coords(year=year)
    ds_list.append(da.to_dataset())
    
passive_ds = xr.concat(ds_list, dim='year').sortby('year')
passive_ds['melt_onset_mean'] = passive_ds['melt_onset_dayofyear'].mean(dim='year')
passive_ds['melt_onset_median'] = passive_ds['melt_onset_dayofyear'].median(dim='year')
passive_ds['melt_onset_std'] = passive_ds['melt_onset_dayofyear'].std(dim='year')
passive_ds['melt_onset_anomaly'] = passive_ds['melt_onset_dayofyear'] - passive_ds['melt_onset_median']
mask_file = glob.glob(f"data/Main_Melt_Onset_Dates_1841/data/*mask*.tif")
mask_da = rxr.open_rasterio(mask_file[0], masked=True).squeeze()
passive_ds["mask"] = mask_da
passive_ds

['passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2011Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2018Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.1995Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.1996Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.1989Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2019Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2004Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2023Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2007Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.1991Ah000-001v000-001.001.tif',
 'passive_data/Main_Melt_Onset_Dates_1841/data/ABoVE.MMOD.2022Ah000-001v000-001.001.tif',
 'passive_

In [7]:
global_ds = config.open_runoff_onset_dataset()
global_ds

<xarray.Dataset> Size: 14TB
Dimensions:                     (water_year: 10, latitude: 195970,
                                 longitude: 499998)
Coordinates:
  * water_year                  (water_year) int64 80B 2015 2016 ... 2023 2024
  * latitude                    (latitude) float64 2MB 81.1 81.1 ... -60.0 -60.0
  * longitude                   (longitude) float64 4MB -180.0 -180.0 ... 180.0
    spatial_ref                 int32 4B ...
Data variables:
    runoff_onset                (water_year, latitude, longitude) float32 4TB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    runoff_onset_mad            (latitude, longitude) float64 784GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    runoff_onset_median         (latitude, longitude) float32 392GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    temporal_resolution         (water_year, latitude, longitude) float64 8TB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    temporal_resolution_median  (latitude, longitude) float64 784GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
Attributes:
    processed_tiles:  []

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
alaska_range_gdf = gmba_gdf[gmba_gdf["MapName"]=="Alaska Range"]
xmin_4326, ymin_4326, xmax_4326, ymax_4326 = alaska_range_gdf.total_bounds

alaska_proj = "EPSG:3338"

alaska_range_proj_gdf = alaska_range_gdf.to_crs(alaska_proj)
xmin_proj, ymin_proj, xmax_proj, ymax_proj = alaska_range_proj_gdf.total_bounds
alaska_range_proj_gdf

,GMBA_V2_ID,GMBA_V1_ID,MapName,WikiDataUR,MapUnit,Hier_Lvl,Feature,Area,Perimeter,Elev_Low,...,Name_ES,Name_FR,Name_PT,Name_RU,Name_ZH,LocalNames,ColorAll,ColorBasic,Color300,geometry
97,11113,3.01.02,Alaska Range,https://www.wikidata.org/wiki/Q156684,Aggregated,4,Mountain range with well-recognized name,78722.45882,11358.71801,29.0,...,Cordillera de Alaska,Chaîne d'Alaska,Cordilheira do Alasca,Аляскинский хребет,阿拉斯加山脈,NaN,5,0,3,"MULTIPOLYGON (((-151.15431 61.76102, -151.1466..."


In [ ]:
states_gdf = gpd.read_file('http://eric.clst.org/assets/wiki/uploads/Stuff/gz_2010_us_040_00_5m.json')
alaska_gdf = states_gdf[states_gdf['NAME']=='Alaska'].clip_by_rect(-175, 0, 0, 89)
alaska_proj_gdf = alaska_gdf.to_crs(alaska_proj)
hillshade_proj_da = rxr.open_rasterio('../../visualize/data/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().rio.clip_box(*alaska_gdf.total_bounds,crs=alaska_gdf.crs).rio.reproject(alaska_proj)#.coarsen(x=4, y=4,boundary='trim').mean()#.compute()
hillshade_proj_da

In [ ]:
alaska_passive_onset_2020_proj_da = passive_ds['melt_onset_dayofyear'].sel(year=2020).rio.reproject(alaska_proj_gdf.crs)#.rio.clip(alaska_proj_gdf.geometry)#.plot.imshow()
alaska_passive_onset_2020_proj_da

<xarray.Dataset> Size: 233MB
Dimensions:               (x: 904, y: 848, year: 36)
Coordinates:
  * x                     (x) float64 7kB -2.872e+06 -2.866e+06 ... 2.779e+06
  * y                     (y) float64 7kB 4.005e+06 3.998e+06 ... -1.296e+06
  * year                  (year) int64 288B 1988 1989 1990 ... 2021 2022 2023
    band                  int64 8B 1
    spatial_ref           int64 8B 0
Data variables:
    melt_onset_dayofyear  (year, y, x) float32 110MB nan nan nan ... nan nan nan
    melt_onset_mean       (y, x) float32 3MB nan nan nan nan ... nan nan nan nan
    melt_onset_median     (y, x) float32 3MB nan nan nan nan ... nan nan nan nan
    melt_onset_std        (y, x) float32 3MB nan nan nan nan ... nan nan nan nan
    melt_onset_anomaly    (year, y, x) float32 110MB nan nan nan ... nan nan nan
    mask                  (y, x) float32 3MB nan nan nan nan ... nan nan nan nan

In [ ]:
alaska_runoff_onset_2020_proj_da = global_ds['runoff_onset'].sel(water_year=2020).rio.clip_box(xmin_4326-2.2, ymin_4326-2.2, xmax_4326+3, ymax_4326+3, crs="EPSG:4326").coarsen(latitude=4,longitude=4, boundary='trim').mean().rio.reproject(alaska_proj)
alaska_runoff_onset_2020_proj_da

<xarray.DataArray 'melt_onset_dayofyear' (y: 848, x: 904)> Size: 3MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(848, 904), dtype=float32)
Coordinates:
  * y            (y) float64 7kB 4.005e+06 3.998e+06 ... -1.29e+06 -1.296e+06
  * x            (x) float64 7kB -2.872e+06 -2.866e+06 ... 2.773e+06 2.779e+06
    band         int64 8B 1
    year         int64 8B 2020
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0

In [ ]:
differences_proj_da = alaska_runoff_onset_2020_proj_da - (passive_ds['melt_onset_dayofyear'].sel(year=2020).rio.reproject_match(alaska_runoff_onset_2020_proj_da)+92)
differences_proj_da

In [ ]:
kennicott_gdf = gpd.read_file("geometries/kennicott.geojson")
kennicott_proj_gdf = kennicott_gdf.to_crs(alaska_proj)
kennicott_proj_gdf

In [ ]:
xmin_kennicott_4326, ymin_kennicott_4326, xmax_kennicott_4326, ymax_kennicott_4326 = kennicott_gdf.total_bounds
xmin_kennicott, ymin_kennicott, xmax_kennicott, ymax_kennicott = kennicott_proj_gdf.total_bounds

In [ ]:
denali_runoff_onset_2020_da = global_ds['runoff_onset'].sel(water_year=2020).rio.clip_box(xmin_denali_4326, ymin_denali_4326, xmax_denali_4326, ymax_denali_4326, crs="EPSG:4326").rio.reproject("EPSG:3338")
denali_runoff_onset_2020_da

In [ ]:
denali_melt_onset_2020_da = alaska_melt_onset_2020_da.rio.clip(denali_proj_gdf.geometry)
denali_melt_onset_2020_da

In [ ]:
denali_differences_da = denali_runoff_onset_2020_da - (denali_melt_onset_2020_da.rio.reproject_match(denali_runoff_onset_2020_da)+92)
denali_differences_da

In [ ]:
s2 = easysnowdata.remote_sensing.Sentinel2(bbox_input=denali_gdf,
                                           start_date="2020-04-10",
                                          end_date="2020-05-10",
                                          resolution=40)
s2.get_rgb()
s2.rgb

In [ ]:
s2_rgb_da = s2.rgb.compute()
s2_rgb_da

In [ ]:
s2_rgb_da.plot.imshow(col='time', col_wrap=5,robust=True)

In [ ]:
s2_rgb_da.sel(time="2020-05-10").squeeze().plot.imshow(vmin=-20,vmax=180)

In [ ]:
s2_rgb_da.sel(time="2020-05-10").squeeze()

In [ ]:
f,axs=plt.subplots(4,1, figsize=(8,13),sharex=True,sharey=True,layout='constrained')
date_vmin = "2020-03-06"
date_vmax = "2020-09-01"
doy_vmin = pd.to_datetime(date_vmin).dayofyear
doy_vmax = pd.to_datetime(date_vmax).dayofyear
dowy_vmin = easysnowdata.utils.datetime_to_DOWY(date_vmin)
dowy_vmax = easysnowdata.utils.datetime_to_DOWY(date_vmax)

print(f'for vmin date {date_vmin}, doy={doy_vmin}, dowy={dowy_vmin}')
print(f'for vmax date {date_vmax}, doy={doy_vmax}, dowy={dowy_vmax}')

# First panel: context map with basemap
#alaska_range_proj_gdf.boundary.plot(ax=axs[0], edgecolor='black', linewidth=1)
denali_proj_gdf.boundary.plot(ax=axs[0], edgecolor='red', linewidth=1)

s2_rgb_da.sel(time="2020-05-10").squeeze().transpose("band", "y", "x").rio.reproject(alaska_proj).plot.imshow(ax=axs[0], vmin=-20, vmax=180)
axs[0].set_aspect('equal')

axs[0].set_xlabel(f"May 10, 2020")
axs[0].set_ylabel("")
axs[0].axis("off")
axs[0].set_title('Alaska Range study area')
# axs[0].set_xlim([xmin_denali, xmax_denali])
# axs[0].set_ylim([ymin_denali, ymax_denali])

# Second through fourth panels: data maps
denali_melt_onset_2020_da.plot.imshow(ax=axs[1], cmap='viridis', vmin=doy_vmin, vmax=doy_vmax, add_colorbar=False,zorder=1)
denali_runoff_onset_2020_da.plot.imshow(ax=axs[2], cmap='viridis', vmin=dowy_vmin, vmax=dowy_vmax, add_colorbar=False,zorder=1)

denali_differences_da.plot.imshow(ax=axs[3], cmap='RdBu', vmin=-75, vmax=75, add_colorbar=False,zorder=1)

for ax in axs[1:4]:
    # ax.set_xlim([xmin_denali, xmax_denali])
    # ax.set_ylim([ymin_denali, ymax_denali])
    denali_proj_gdf.boundary.plot(ax=ax, edgecolor='red', linewidth=1)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.axis("off")
    ax.set_aspect('equal')

    
axs[1].set_title('Passive microwave melt onset')
axs[2].set_title('Our runoff onset product')
axs[3].set_title('Difference map (Our product - Passive)')

f.suptitle('Comparison of snowmelt timing for the Alaska Range in 2020')
#f.savefig('figures/passive_comparison/alaska_runoff_onset_vs_passive_2020_4panel.png', dpi=300)

In [19]:
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as path_effects
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import ConnectionPatch
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import box as shapely_box

# --- Color / date range setup ---
date_vmin = "2020-01-18"
date_vmax = "2020-06-26"
doy_vmin  = pd.to_datetime(date_vmin).dayofyear
doy_vmax  = pd.to_datetime(date_vmax).dayofyear
dowy_vmin = easysnowdata.utils.datetime_to_DOWY(date_vmin)
dowy_vmax = easysnowdata.utils.datetime_to_DOWY(date_vmax)

print(f'for vmin date {date_vmin}, doy={doy_vmin}, dowy={dowy_vmin}')
print(f'for vmax date {date_vmax}, doy={doy_vmax}, dowy={dowy_vmax}')

xmin_4326, ymin_4326, xmax_4326, ymax_4326 = alaska_range_gdf.total_bounds

# Month slots for viridis colorbar (DOY scale, 2020)
_all_month_doys  = [pd.Timestamp(f'2020-{m:02d}-01').dayofyear for m in range(1, 13)]
_all_month_names = ['JAN','FEB','MAR','APR','MAY','JUN',
                    'JUL','AUG','SEP','OCT','NOV','DEC']
_all_month_ends  = _all_month_doys[1:] + [367]

visible_months = []
boundary_doys  = []
for ms, me, name in zip(_all_month_doys, _all_month_ends, _all_month_names):
    if me <= doy_vmin or ms >= doy_vmax:
        continue
    vis_start = max(ms, doy_vmin)
    vis_end   = min(me, doy_vmax)
    visible_months.append(dict(center=(vis_start + vis_end) / 2,
                               width=vis_end - vis_start,
                               name=name))
    if doy_vmin < ms < doy_vmax:
        boundary_doys.append(ms)

# Shared style constants
TITLE_FS = 28
STROKE   = [path_effects.withStroke(linewidth=3, foreground='black')]

# ── Helper: hillshade + bounds + boundary for data maps ───────────────────
def _style_map_ax(ax):
    hillshade_da.plot.imshow(ax=ax, cmap='gray', vmin=0, vmax=255, add_colorbar=False, zorder=0)
    alaska_range_proj_gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=1.5, zorder=2)
    ax.set_xlim([xmin - 1e5, xmax + 1e5])
    ax.set_ylim([ymin - 1e5, ymax + 1e5])
    ax.set_aspect('equal')
    ax.axis('off')

def _map_title(ax, text):
    ax.text(0.5, 0.98, text, transform=ax.transAxes,
            ha='center', va='top', color='white', fontname='Oswald',
            fontsize=TITLE_FS, fontweight='bold', path_effects=STROKE)

# --- Figure / GridSpec ---
fig = plt.figure(figsize=(22, 10))
# Increase wspace so the histogram left y-axis has room and isn't behind the map
outer_gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[1, 1.6, 1.6],
                              wspace=0.04, left=0.01, right=0.99, bottom=0.03, top=0.97)

# Globe fills the entire left column
ax_globe = fig.add_subplot(outer_gs[0, 0],
                            projection=ccrs.Orthographic(central_longitude=-150, central_latitude=64))
# Satellite overlaid on bottom portion of globe (drawn on top via later add_axes call)
left_bbox = outer_gs[0, 0].get_position(fig)
sat_frac  = 0.55   # fraction of left column height the satellite covers
ax_sat = fig.add_axes([left_bbox.x0,
                        left_bbox.y0,
                        left_bbox.width,
                        left_bbox.height * sat_frac])

# Thicker colorbar rows (0.07 → 0.16) and more hspace so histogram doesn't overlap colorbar
mid_gs     = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer_gs[0, 1],
                                               height_ratios=[1, 0.1, 1], hspace=0.04)
ax_passive = fig.add_subplot(mid_gs[0])
ax_cbar    = fig.add_subplot(mid_gs[1])
ax_ours    = fig.add_subplot(mid_gs[2])

right_gs     = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer_gs[0, 2],
                                                 height_ratios=[1, 0.1, 1], hspace=0.04)
ax_diff      = fig.add_subplot(right_gs[0])
ax_diff_cbar = fig.add_subplot(right_gs[1])
ax_hist      = fig.add_subplot(right_gs[2])

# Nudge histogram: shift left edge right (clears map) and shrink from top (clears colorbar)
_p = ax_hist.get_position()
ax_hist.set_position([_p.x0 + 0.025, _p.y0, _p.width - 0.025, _p.height - 0.035])

# ── 1. Orthographic globe ──────────────────────────────────────────────────
ax_globe.set_global()
ax_globe.add_feature(cfeature.LAND, zorder=0, edgecolor='gray', linewidth=0.3)
ax_globe.add_feature(cfeature.OCEAN, zorder=0)
ax_globe.add_geometries(alaska_range_gdf.geometry, crs=ccrs.PlateCarree(),
                         edgecolor='black', facecolor='none', linewidth=0.3, zorder=2)
bbox_4326 = shapely_box(xmin_4326 - 1, ymin_4326 - 1, xmax_4326 + 1, ymax_4326 + 1)
ax_globe.add_geometries([bbox_4326], crs=ccrs.PlateCarree(),
                         edgecolor='red', facecolor='none', linewidth=1, zorder=3)

# ── 2. Satellite / basemap context ────────────────────────────────────────
alaska_range_proj_gdf.boundary.plot(ax=ax_sat, edgecolor='black', linewidth=1.5, zorder=2)
ctx.add_basemap(ax_sat, crs=alaska_range_proj_gdf.crs.to_epsg(),
                source=ctx.providers.Esri.WorldImagery, attribution='', zoom=6)
ax_sat.set_xlim([xmin - 1e5, xmax + 1e5])
ax_sat.set_ylim([ymin - 1e5, ymax + 1e5])
ax_sat.set_aspect('equal')
ax_sat.axis('off')
scalebar = ScaleBar(1, 'm', fixed_value=150, fixed_units='km', location='upper right',
                    color='white', box_color='none', box_alpha=0,
                    font_properties={'size': 14, 'weight': 'bold'})
ax_sat.add_artist(scalebar)
_map_title(ax_sat, 'Alaska Range')
ax_sat.set_title("")

# ── Red connecting lines: globe bbox bottom corners → satellite top corners ─
pt_ll = ax_globe.projection.transform_point(xmin_4326 - 1, ymin_4326 - 1, ccrs.PlateCarree())
pt_lr = ax_globe.projection.transform_point(xmax_4326 + 1, ymin_4326 - 1, ccrs.PlateCarree())
for xyA, xyB in [(pt_ll, (xmin - 1e5, ymax + 1e5)),
                 (pt_lr, (xmax + 1e5, ymax + 1e5))]:
    fig.add_artist(ConnectionPatch(xyA=xyA, coordsA=ax_globe.transData,
                                   xyB=xyB, coordsB=ax_sat.transData,
                                   color='red', linewidth=1, zorder=10))

# ── 3. Passive microwave melt onset ───────────────────────────────────────
_style_map_ax(ax_passive)
alaska_melt_onset_2020_da.plot.imshow(ax=ax_passive, cmap='viridis',
                                       vmin=doy_vmin, vmax=doy_vmax,
                                       add_colorbar=False, zorder=1, alpha=0.85)
_map_title(ax_passive, 'Passive microwave melt onset, 2020')
ax_passive.set_title("")

# ── 4. Shared viridis colorbar (month-slot labels, dashed boundaries) ──────
norm_viridis = mcolors.Normalize(vmin=doy_vmin, vmax=doy_vmax)
cbar_v = plt.colorbar(cm.ScalarMappable(norm=norm_viridis, cmap='viridis'),
                       cax=ax_cbar, orientation='horizontal', extend='both')
cbar_v.set_ticks([])
for b in boundary_doys:
    ax_cbar.axvline(x=b, color='white', linestyle='--', linewidth=1.5, zorder=10)
total_doy_range = doy_vmax - doy_vmin
for m in visible_months:
    x_frac = (m['center'] - doy_vmin) / total_doy_range
    t = ax_cbar.text(x_frac, 0.5, m['name'], fontsize=16,
                     ha='center', va='center', color='white', fontweight='bold',
                     transform=ax_cbar.transAxes, clip_on=True)
    t.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

# ── 5. Our runoff onset product ───────────────────────────────────────────
_style_map_ax(ax_ours)
alaska_runoff_onset_2020_da.plot.imshow(ax=ax_ours, cmap='viridis',
                                         vmin=dowy_vmin, vmax=dowy_vmax,
                                         add_colorbar=False, zorder=1, alpha=0.85)
_map_title(ax_ours, 'Our runoff onset product, 2020')
ax_ours.set_title("")

# ── 6. Difference map ─────────────────────────────────────────────────────
norm_diff = mcolors.Normalize(vmin=-75, vmax=75)
_style_map_ax(ax_diff)
differences_da.plot.imshow(ax=ax_diff, cmap='RdBu', vmin=-75, vmax=75,
                            add_colorbar=False, zorder=1, alpha=0.85)
_map_title(ax_diff, 'Difference map (our product - passive)')
ax_diff.set_title("")

# ── 7. Diverging colorbar ─────────────────────────────────────────────────
cbar_d = plt.colorbar(cm.ScalarMappable(norm=norm_diff, cmap='RdBu'),
                       cax=ax_diff_cbar, orientation='horizontal', extend='both')
cbar_d.set_ticks([-75, -50, -25, 0, 25, 50, 75])
cbar_d.ax.tick_params(labelsize=14)
ax_diff_cbar.set_xlabel('Difference [days]', fontsize=14, labelpad=2)
# Lower y to 0.22 so text sits in the bottom portion of the colorbar
for label, ha, x in [('Our product earlier', 'left', 0.02),
                      ('Our product later',   'right', 0.98)]:
    ax_diff_cbar.text(x, 0.5, label, transform=ax_diff_cbar.transAxes,
                      ha=ha, va='center', color='white', fontweight='bold', fontsize=16,
                      path_effects=[path_effects.withStroke(linewidth=2, foreground='black')])

# ── 8. Histogram ──────────────────────────────────────────────────────────
diff_vals = differences_da.values.flatten()
diff_vals = diff_vals[~np.isnan(diff_vals)]
n_h, bins_h, patches_h = ax_hist.hist(diff_vals, bins=50, edgecolor='black', linewidth=0.5)
cmap_rdbu = cm.get_cmap('RdBu')
for patch, l, r in zip(patches_h, bins_h[:-1], bins_h[1:]):
    patch.set_facecolor(cmap_rdbu(norm_diff((l + r) / 2)))
ax_hist.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax_hist.set_xlim([-150, 150])
ax_hist.tick_params(axis='both', which='major', labelsize=14, length=6, width=1.2)
ylim_h = ax_hist.get_ylim()
ax_hist.text(-145, ylim_h[1] * 0.3, 'Our product earlier',
             color='darkred',  fontsize=16, fontweight='bold', ha='left')
ax_hist.text( 145, ylim_h[1] * 0.3, 'Our product later',
             color='darkblue', fontsize=16, fontweight='bold', ha='right')
median_diff = np.nanmedian(diff_vals)
mad_diff    = np.nanmedian(np.abs(diff_vals - median_diff))
# Lowered to 0.55 and bigger font
ax_hist.text(145, ylim_h[1] * 0.9,
             f'Median difference: {median_diff:.1f} days\nMAD: {mad_diff:.1f} days',
             fontsize=13, ha='right', va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax_hist.set_xlabel('Difference [days]', fontsize=14)
ax_hist.set_ylabel('Count', fontsize=14)
ax_hist.spines['top'].set_visible(False)
ax_hist.spines['right'].set_visible(False)

fig.savefig(FIGURE_DIR / 'alaska_comparison_multipanel_v2.png', dpi=300, bbox_inches='tight')


for vmin date 2020-01-18, doy=18, dowy=110
for vmax date 2020-06-26, doy=178, dowy=270


/tmp/ipykernel_2385624/4230083690.py:183: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_rdbu = cm.get_cmap('RdBu')
